# External-Data Weight Optimization for a Three-Model CXR Ensemble

This notebook calculates non-negative soft-voting weights for three chest X-ray classifiers:

- EfficientNet-B1 with final Coordinate Attention
- Swin Transformer Tiny
- DenseNet121 with Coordinate Attention

The weights are learned only from the strict external manifest. The original dataset test split is intentionally not evaluated here; it must remain untouched for the final evaluation after the weights are frozen.


## Method and leakage boundary

For each external image, every model produces a four-class probability vector. The ensemble probability is

\[
p_{ens}=w_{eff}p_{eff}+w_{swin}p_{swin}+w_{dense}p_{dense},
\]

where every weight is non-negative and the three weights sum to one. The optimizer minimizes class-balanced multiclass log loss. This objective uses probability quality, remains suitable for unequal class counts, and gives a stable convex weight-fitting problem.

The metrics printed in this notebook are calibration-data diagnostics, not final test results. After this notebook finishes, freeze the saved weights and evaluate them once on the untouched original test split.


In [ ]:
# Thư viện chuẩn và xử lý dữ liệu
import os
import gc
import json
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile

# Thư viện học sâu
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# Thư viện đánh giá và tối ưu
from scipy.optimize import minimize
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    classification_report,
    confusion_matrix,
)

# Cho phép PIL đọc một số ảnh bị cắt nhẹ ở cuối tệp
ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)


## Configuration

Attach the three checkpoints, the strict manifest, and the external image dataset to the Kaggle notebook. The file finder supports files placed inside any attached Kaggle dataset or model directory.

`MAX_IMAGES` must remain `None` when calculating the real weights. A small integer may be used only for a quick pipeline test.


In [ ]:
# Thư mục đầu vào và đầu ra trên Kaggle
KAGGLE_INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/ensemble_weight_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Tên tệp đã được kiểm tra từ các tệp người dùng cung cấp
EFF_FILENAMES = ["best_EfficientNet_B1_CA_Last.pth"]
SWIN_FILENAMES = ["best_Swin_Tiny.pth"]
DENSE_FILENAMES = ["densenet121_ca_best(1).pth", "densenet121_ca_best.pth"]
MANIFEST_FILENAMES = ["clean_new_dataset_manifest_strict.csv"]

# Có thể đặt đường dẫn gốc thủ công nếu đường dẫn trong manifest không còn đúng
EXTERNAL_DATASET_ROOT = Path("/kaggle/input/datasets/reflex7/cxr-data-set")

# Dataset gốc: chỉ thư mục test được dùng ở bước đánh giá cuối
ORIGINAL_DATASET_ROOT = Path(
    "/kaggle/input/datasets/jtiptj/chest-xray-pneumoniacovid19tuberculosis"
)

# Cấu hình inference; giảm batch size nếu GPU hết bộ nhớ
BATCH_SIZE = 32
NUM_WORKERS = 2
PIN_MEMORY = torch.cuda.is_available()
USE_AMP = torch.cuda.is_available()
MAX_IMAGES = None  # Chỉ đặt số nhỏ để chạy thử, phải là None khi tính weight thật

# Thứ tự lớp phải giống đầu ra của cả ba checkpoint
MODEL_CLASS_NAMES = ["COVID19", "NORMAL", "PNEUMONIA", "TURBERCULOSIS"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(MODEL_CLASS_NAMES)}

# Manifest dùng cách viết đúng, checkpoint dùng tên TURBERCULOSIS bị sai chính tả
LABEL_ALIASES = {
    "COVID": "COVID19",
    "COVID-19": "COVID19",
    "COVID19": "COVID19",
    "NORMAL": "NORMAL",
    "PNEUMONIA": "PNEUMONIA",
    "TUBERCULOSIS": "TURBERCULOSIS",
    "TURBERCULOSIS": "TURBERCULOSIS",
}

STRICT_KEEP_STATUS = "NO_EXACT_STRONG_OR_POSSIBLE_OVERLAP"


## Locate the attached resources

This section indexes `/kaggle/input` once, locates each required file by name, and discovers possible external dataset roots. Exact checkpoint filenames are used to prevent loading the wrong experiment accidentally.


In [ ]:
def build_input_index(input_root):
    # Quét một lần để tránh tìm lại toàn bộ /kaggle/input cho từng tệp
    files_by_name = defaultdict(list)
    dirs_by_name = defaultdict(list)

    if not input_root.exists():
        raise FileNotFoundError(f"Kaggle input directory does not exist: {input_root}")

    for current_root, dirs, files in os.walk(input_root):
        current_path = Path(current_root)
        dirs_by_name[current_path.name].append(current_path)
        for filename in files:
            files_by_name[filename].append(current_path / filename)

    return files_by_name, dirs_by_name


def locate_one(files_by_name, candidate_names, resource_name):
    # Chỉ chấp nhận đúng một tệp để tránh dùng nhầm checkpoint
    matches = []
    for filename in candidate_names:
        matches.extend(files_by_name.get(filename, []))

    matches = sorted(set(matches))
    if not matches:
        raise FileNotFoundError(
            f"Cannot find {resource_name}. Expected one of: {candidate_names}"
        )
    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple files found for {resource_name}: {matches}. "
            "Remove duplicate attachments or set the path manually."
        )
    return matches[0]


files_by_name, dirs_by_name = build_input_index(KAGGLE_INPUT)

EFF_PATH = locate_one(files_by_name, EFF_FILENAMES, "EfficientNet checkpoint")
SWIN_PATH = locate_one(files_by_name, SWIN_FILENAMES, "Swin checkpoint")
DENSE_PATH = locate_one(files_by_name, DENSE_FILENAMES, "DenseNet checkpoint")
MANIFEST_PATH = locate_one(files_by_name, MANIFEST_FILENAMES, "strict manifest")

print("EfficientNet checkpoint:", EFF_PATH)
print("Swin checkpoint:        ", SWIN_PATH)
print("DenseNet checkpoint:    ", DENSE_PATH)
print("Strict manifest:        ", MANIFEST_PATH)


## Read and validate the strict manifest

Only rows marked `NO_EXACT_STRONG_OR_POSSIBLE_OVERLAP` are accepted. The manifest label `TUBERCULOSIS` is deliberately mapped to the checkpoint label `TURBERCULOSIS`, preserving the output order used during model training.


In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)

# Kiểm tra các cột bắt buộc trước khi xử lý
required_columns = {
    "path", "relative_path", "dataset", "split", "label", "overlap_status"
}
missing_columns = required_columns - set(manifest.columns)
if missing_columns:
    raise ValueError(f"Manifest is missing columns: {sorted(missing_columns)}")

print("Original manifest shape:", manifest.shape)
print("Overlap status counts:")
display(manifest["overlap_status"].value_counts(dropna=False).to_frame("count"))

# Chỉ giữ các ảnh đã vượt qua kiểm tra exact, strong và possible duplicate
clean_df = manifest.loc[
    manifest["overlap_status"].eq(STRICT_KEEP_STATUS)
].copy()

if clean_df.empty:
    raise ValueError("No strict clean images remain after filtering.")

# Chuẩn hóa tên nhãn và ánh xạ sang chỉ số đầu ra của checkpoint
clean_df["label_normalized"] = (
    clean_df["label"].astype(str).str.strip().str.upper().map(LABEL_ALIASES)
)

unknown_labels = clean_df.loc[
    clean_df["label_normalized"].isna(), "label"
].drop_duplicates().tolist()
if unknown_labels:
    raise ValueError(f"Unknown labels in manifest: {unknown_labels}")

clean_df["label_idx"] = clean_df["label_normalized"].map(CLASS_TO_IDX).astype(int)

# Loại dòng đường dẫn lặp nếu có; không thay đổi nội dung ảnh
duplicate_rows = int(clean_df["relative_path"].duplicated().sum())
if duplicate_rows:
    warnings.warn(f"Removing {duplicate_rows} repeated relative_path rows.")
    clean_df = clean_df.drop_duplicates("relative_path", keep="first").copy()

clean_df = clean_df.reset_index(drop=True)

if MAX_IMAGES is not None:
    # Lấy mẫu phân tầng gần đúng chỉ để kiểm tra pipeline
    clean_df = (
        clean_df.groupby("label_idx", group_keys=False)
        .apply(lambda group: group.sample(
            n=min(len(group), max(1, MAX_IMAGES // len(MODEL_CLASS_NAMES))),
            random_state=SEED,
        ))
        .reset_index(drop=True)
    )
    warnings.warn("MAX_IMAGES is active. Do not use these weights as final weights.")

display(clean_df["label_normalized"].value_counts().reindex(MODEL_CLASS_NAMES).to_frame("count"))
print("Strict clean rows used:", len(clean_df))


In [ ]:
def candidate_dataset_roots():
    # Tạo danh sách thư mục có thể chứa relative_path trong manifest
    roots = []
    if EXTERNAL_DATASET_ROOT is not None:
        roots.append(Path(EXTERNAL_DATASET_ROOT))
    roots.extend(dirs_by_name.get("cxr-data-set", []))

    # Giữ nguyên thứ tự nhưng loại thư mục trùng
    unique_roots = []
    seen = set()
    for root in roots:
        key = str(root)
        if key not in seen:
            seen.add(key)
            unique_roots.append(root)
    return unique_roots


DATASET_ROOTS = candidate_dataset_roots()
print("Candidate external dataset roots:")
for root in DATASET_ROOTS:
    print(" -", root, "exists=", root.exists())


def resolve_image_path(row):
    # Ưu tiên đường dẫn tuyệt đối được lưu trong manifest
    original_path = Path(str(row["path"]))
    if original_path.is_file():
        return str(original_path)

    # Nếu Kaggle mount dataset ở vị trí khác, ghép root mới với relative_path
    relative_path = Path(str(row["relative_path"]))
    for root in DATASET_ROOTS:
        candidate = root / relative_path
        if candidate.is_file():
            return str(candidate)
    return None


clean_df["resolved_path"] = clean_df.apply(resolve_image_path, axis=1)
missing_mask = clean_df["resolved_path"].isna()

if missing_mask.any():
    examples = clean_df.loc[missing_mask, ["path", "relative_path"]].head(10)
    display(examples)
    raise FileNotFoundError(
        f"Cannot resolve {int(missing_mask.sum())} image paths. "
        "Attach the external CXR dataset or correct EXTERNAL_DATASET_ROOT."
    )

print("All image paths were resolved successfully.")

# Lưu chính xác danh sách đã dùng để có thể kiểm tra lại thí nghiệm
clean_df.to_csv(OUTPUT_DIR / "clean_manifest_used_for_weights.csv", index=False)


## Model definitions

The definitions below match the parameter names and tensor shapes found in the supplied checkpoints:

- EfficientNet-B1: 1,280-channel final feature map, CA bottleneck of 40 channels, and a 4-class classifier.
- Swin Tiny: torchvision Swin-T with a 4-class linear head.
- DenseNet121: 1,024-channel final feature map, CA bottleneck of 32 channels, and a `1024 → 512 → 4` classifier.


In [ ]:
class HSigmoid(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu = nn.ReLU6(inplace=True)

    def forward(self, x):
        return self.relu(x + 3.0) / 6.0


class HSwish(nn.Module):
    def __init__(self):
        super().__init__()
        self.h_sigmoid = HSigmoid()

    def forward(self, x):
        return x * self.h_sigmoid(x)


class EfficientCoordinateAttention(nn.Module):
    # CA khớp với các key att_last.* trong checkpoint EfficientNet

    def __init__(self, channels=1280, reduction=32):
        super().__init__()
        hidden = max(8, channels // reduction)  # 1280 // 32 = 40
        self.conv_reduce = nn.Conv2d(channels, hidden, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(hidden)
        self.act = HSwish()
        self.conv_h = nn.Conv2d(hidden, channels, kernel_size=1, bias=False)
        self.conv_w = nn.Conv2d(hidden, channels, kernel_size=1, bias=False)

    def forward(self, x):
        identity = x
        height, width = x.shape[2], x.shape[3]

        # Gom thông tin riêng theo chiều cao và chiều rộng
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)

        y = torch.cat([x_h, x_w], dim=2)
        y = self.act(self.bn(self.conv_reduce(y)))
        y_h, y_w = torch.split(y, [height, width], dim=2)
        y_w = y_w.permute(0, 1, 3, 2)

        attention_h = torch.sigmoid(self.conv_h(y_h))
        attention_w = torch.sigmoid(self.conv_w(y_w))
        return identity * attention_h * attention_w


class EfficientNetB1CALast(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        base = models.efficientnet_b1(weights=None)
        self.features = base.features
        self.avgpool = base.avgpool
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2, inplace=True),
            nn.Linear(1280, num_classes),
        )
        self.att_last = EfficientCoordinateAttention(1280, reduction=32)

    def forward(self, x):
        x = self.features(x)
        x = self.att_last(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


In [ ]:
class SwinTinyClassifier(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.model = models.swin_t(weights=None)
        in_features = self.model.head.in_features
        self.model.head = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.model(x)


class DenseCoordinateAttention(nn.Module):
    # CA khớp với các key coordinate_attention.* trong checkpoint DenseNet

    def __init__(self, channels=1024, reduction=32):
        super().__init__()
        hidden = max(8, channels // reduction)  # 1024 // 32 = 32
        self.conv1 = nn.Conv2d(channels, hidden, kernel_size=1, bias=False)
        self.batch_norm = nn.BatchNorm2d(hidden)
        self.h_swish = HSwish()
        self.conv_h = nn.Conv2d(hidden, channels, kernel_size=1, bias=True)
        self.conv_w = nn.Conv2d(hidden, channels, kernel_size=1, bias=True)

    def forward(self, x):
        identity = x
        height, width = x.shape[2], x.shape[3]

        # Tạo hai nhánh attention theo tọa độ
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
        y = torch.cat([x_h, x_w], dim=2)
        y = self.h_swish(self.batch_norm(self.conv1(y)))
        y_h, y_w = torch.split(y, [height, width], dim=2)
        y_w = y_w.permute(0, 1, 3, 2)

        attention_h = torch.sigmoid(self.conv_h(y_h))
        attention_w = torch.sigmoid(self.conv_w(y_w))
        return identity * attention_h * attention_w


class DenseNet121WithCA(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        base = models.densenet121(weights=None)
        self.features = base.features
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )
        self.coordinate_attention = DenseCoordinateAttention(1024, reduction=32)

    def forward(self, x):
        x = self.features(x)
        x = F.relu(x, inplace=True)
        x = self.coordinate_attention(x)
        x = F.adaptive_avg_pool2d(x, output_size=(1, 1))
        x = torch.flatten(x, 1)
        return self.classifier(x)


## Strict checkpoint loading

Every checkpoint is loaded with `weights_only=True`. Strict state-dictionary loading is required: a missing or unexpected layer stops the notebook instead of silently producing invalid predictions.

The EfficientNet checkpoint does not store class names. Its output order is therefore explicitly assumed to match the other two supplied checkpoints: `COVID19, NORMAL, PNEUMONIA, TURBERCULOSIS`. Confirm this against the EfficientNet training notebook before treating the weights as final.


In [ ]:
def load_checkpoint_safely(path):
    # weights_only tránh thực thi đối tượng pickle tùy ý trong checkpoint
    return torch.load(path, map_location="cpu", weights_only=True)


def load_strict(model, checkpoint_path, state_key, model_name):
    checkpoint = load_checkpoint_safely(checkpoint_path)
    if state_key not in checkpoint:
        raise KeyError(
            f"{model_name}: checkpoint does not contain key '{state_key}'. "
            f"Available keys: {list(checkpoint.keys())}"
        )

    state_dict = checkpoint[state_key]
    incompatibility = model.load_state_dict(state_dict, strict=True)
    if incompatibility.missing_keys or incompatibility.unexpected_keys:
        raise RuntimeError(
            f"{model_name}: incompatible checkpoint. "
            f"Missing={incompatibility.missing_keys}, "
            f"unexpected={incompatibility.unexpected_keys}"
        )
    print(f"Loaded {model_name} strictly from {checkpoint_path.name}")
    return checkpoint


eff_model = EfficientNetB1CALast(num_classes=4)
swin_model = SwinTinyClassifier(num_classes=4)
dense_model = DenseNet121WithCA(num_classes=4)

eff_checkpoint = load_strict(
    eff_model, EFF_PATH, "state_dict", "EfficientNet-B1 + CA"
)
swin_checkpoint = load_strict(
    swin_model, SWIN_PATH, "model_state_dict", "Swin Tiny"
)
dense_checkpoint = load_strict(
    dense_model, DENSE_PATH, "model_state_dict", "DenseNet121 + CA"
)

# Kiểm tra metadata lớp của hai checkpoint có lưu thông tin này
for name, checkpoint in [
    ("Swin Tiny", swin_checkpoint),
    ("DenseNet121 + CA", dense_checkpoint),
]:
    stored_classes = checkpoint.get("class_names")
    if stored_classes is not None and list(stored_classes) != MODEL_CLASS_NAMES:
        raise ValueError(
            f"{name} class order mismatch: {stored_classes} != {MODEL_CLASS_NAMES}"
        )

print("EfficientNet metadata:", {
    "mode": eff_checkpoint.get("mode"),
    "epoch": eff_checkpoint.get("epoch"),
    "val_f1": eff_checkpoint.get("val_f1"),
})
print("Swin metadata:", {
    "model_name": swin_checkpoint.get("model_name"),
    "epoch": swin_checkpoint.get("epoch"),
    "val_f1_macro": swin_checkpoint.get("val_f1_macro"),
    "image_size": swin_checkpoint.get("image_size"),
})
print("DenseNet metadata:", {
    "architecture": dense_checkpoint.get("architecture"),
    "epoch": dense_checkpoint.get("epoch"),
    "best_val_macro_f1": dense_checkpoint.get("best_val_macro_f1"),
    "image_size": dense_checkpoint.get("image_size"),
})


## Deterministic preprocessing

Inference uses no random augmentation. EfficientNet-B1 and Swin Tiny use 240×240 inputs, while DenseNet121 uses the 300×300 size stored in its checkpoint. All three use ImageNet normalization, consistent with torchvision pretrained backbones.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def make_transform(image_size):
    # Không dùng augmentation khi tính xác suất cho ensemble
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


MODEL_SPECS = {
    "efficientnet": {
        "model": eff_model,
        "image_size": 240,
        "transform": make_transform(240),
    },
    "swin": {
        "model": swin_model,
        "image_size": int(swin_checkpoint.get("image_size", 240)),
        "transform": make_transform(int(swin_checkpoint.get("image_size", 240))),
    },
    "densenet_ca": {
        "model": dense_model,
        "image_size": int(dense_checkpoint.get("image_size", 300)),
        "transform": make_transform(int(dense_checkpoint.get("image_size", 300))),
    },
}

display(pd.DataFrame([
    {"model": name, "image_size": spec["image_size"]}
    for name, spec in MODEL_SPECS.items()
]))


In [ ]:
class ExternalCXRDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        try:
            image = Image.open(row["resolved_path"]).convert("RGB")
        except Exception as error:
            raise RuntimeError(
                f"Cannot read image at row {index}: {row['resolved_path']}"
            ) from error

        image = self.transform(image)
        return image, int(row["label_idx"]), index


def make_loader(transform):
    dataset = ExternalCXRDataset(clean_df, transform)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
    )


## Generate and cache model probabilities

Models run sequentially to control GPU memory. Probabilities are cached immediately after each model finishes, allowing the weight-optimization cells to be rerun without repeating inference in the same Kaggle output directory.


In [ ]:
def predict_probabilities(model_name, model, transform):
    cache_path = OUTPUT_DIR / f"{model_name}_probabilities.npy"

    # Dùng cache chỉ khi số dòng và số lớp đều khớp
    if cache_path.exists():
        cached = np.load(cache_path)
        if cached.shape == (len(clean_df), len(MODEL_CLASS_NAMES)):
            print(f"Loaded cached probabilities: {cache_path.name}")
            return cached.astype(np.float64)
        warnings.warn(f"Ignoring incompatible cache: {cache_path}")

    loader = make_loader(transform)
    model = model.to(DEVICE)
    model.eval()
    probabilities = np.zeros(
        (len(clean_df), len(MODEL_CLASS_NAMES)), dtype=np.float32
    )

    with torch.inference_mode():
        for batch_number, (images, _, indices) in enumerate(loader, start=1):
            images = images.to(DEVICE, non_blocking=True)

            # AMP giúp giảm bộ nhớ và tăng tốc inference trên GPU
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                logits = model(images)

            batch_probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
            probabilities[indices.numpy()] = batch_probs

            if batch_number % 100 == 0 or batch_number == len(loader):
                print(
                    f"{model_name}: batch {batch_number}/{len(loader)} "
                    f"({min(batch_number * BATCH_SIZE, len(clean_df))}/{len(clean_df)} images)"
                )

    # Kiểm tra xác suất trước khi lưu
    if not np.isfinite(probabilities).all():
        raise ValueError(f"{model_name} produced NaN or infinite probabilities.")
    if not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-4):
        raise ValueError(f"{model_name} probabilities do not sum to one.")

    np.save(cache_path, probabilities)
    print(f"Saved probabilities: {cache_path}")

    # Giải phóng GPU trước khi chạy mô hình tiếp theo
    model.to("cpu")
    del loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return probabilities.astype(np.float64)


all_probabilities = {}
for model_name, spec in MODEL_SPECS.items():
    print(f"\nRunning {model_name}...")
    all_probabilities[model_name] = predict_probabilities(
        model_name=model_name,
        model=spec["model"],
        transform=spec["transform"],
    )

y_true = clean_df["label_idx"].to_numpy(dtype=np.int64)
print("Probability generation completed.")


## Individual-model diagnostics

These values describe performance on the external weight-fitting dataset. They help detect a wrong class order or preprocessing mismatch before optimization. They are not final test metrics.


In [ ]:
def metric_row(name, probabilities, y_true, sample_weight=None):
    predictions = probabilities.argmax(axis=1)
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, predictions),
        "macro_f1": f1_score(y_true, predictions, average="macro"),
        "weighted_f1": f1_score(y_true, predictions, average="weighted"),
        "log_loss": log_loss(
            y_true,
            probabilities,
            labels=np.arange(len(MODEL_CLASS_NAMES)),
            sample_weight=sample_weight,
        ),
    }


individual_rows = [
    metric_row(name, probs, y_true)
    for name, probs in all_probabilities.items()
]
individual_metrics = pd.DataFrame(individual_rows).sort_values(
    "macro_f1", ascending=False
)
display(individual_metrics.style.format({
    "accuracy": "{:.4f}",
    "macro_f1": "{:.4f}",
    "weighted_f1": "{:.4f}",
    "log_loss": "{:.4f}",
}))

# Cảnh báo mạnh nếu một mô hình gần mức đoán ngẫu nhiên, thường do sai thứ tự lớp
if individual_metrics["accuracy"].min() < 0.35:
    warnings.warn(
        "At least one model has very low external accuracy. Verify its class order, "
        "architecture, and preprocessing before optimizing final weights."
    )


## Optimize class-balanced soft-voting weights

Each image receives a class-balancing sample weight inversely proportional to its class frequency. SLSQP enforces the simplex constraints directly. Because log loss of a linear probability mixture is convex in the mixture weights, multiple starting points are used as a numerical consistency check rather than as a search for disconnected local optima.


In [ ]:
MODEL_ORDER = ["efficientnet", "swin", "densenet_ca"]
probability_stack = np.stack(
    [all_probabilities[name] for name in MODEL_ORDER], axis=0
)

# Trọng số mẫu giúp bốn lớp đóng góp ngang nhau vào hàm mục tiêu
class_counts = np.bincount(y_true, minlength=len(MODEL_CLASS_NAMES))
if np.any(class_counts == 0):
    raise ValueError(f"Every class must be present. Counts: {class_counts.tolist()}")

class_balance_weights = len(y_true) / (
    len(MODEL_CLASS_NAMES) * class_counts.astype(np.float64)
)
sample_weights = class_balance_weights[y_true]


def blend_probabilities(weights):
    # Cộng xác suất của ba mô hình theo weight
    blended = np.tensordot(weights, probability_stack, axes=(0, 0))
    return np.clip(blended, 1e-12, 1.0)


def balanced_log_loss_objective(weights):
    return log_loss(
        y_true,
        blend_probabilities(weights),
        labels=np.arange(len(MODEL_CLASS_NAMES)),
        sample_weight=sample_weights,
    )


constraints = ({"type": "eq", "fun": lambda weights: weights.sum() - 1.0},)
bounds = [(0.0, 1.0)] * len(MODEL_ORDER)

# Nhiều điểm bắt đầu để kiểm tra độ ổn định số của nghiệm
starting_points = [
    np.array([1/3, 1/3, 1/3], dtype=np.float64),
    np.array([0.8, 0.1, 0.1], dtype=np.float64),
    np.array([0.1, 0.8, 0.1], dtype=np.float64),
    np.array([0.1, 0.1, 0.8], dtype=np.float64),
]

optimization_results = []
for start in starting_points:
    result = minimize(
        balanced_log_loss_objective,
        x0=start,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 500, "ftol": 1e-12, "disp": False},
    )
    optimization_results.append(result)

successful_results = [result for result in optimization_results if result.success]
if not successful_results:
    messages = [result.message for result in optimization_results]
    raise RuntimeError(f"Weight optimization failed: {messages}")

best_result = min(successful_results, key=lambda result: result.fun)
optimized_weights = np.clip(best_result.x, 0.0, 1.0)
optimized_weights = optimized_weights / optimized_weights.sum()

print("Optimization message:", best_result.message)
print("Balanced log loss:", balanced_log_loss_objective(optimized_weights))
for name, weight in zip(MODEL_ORDER, optimized_weights):
    print(f"{name:15s}: {weight:.8f}")
print("Weight sum:", optimized_weights.sum())


## Compare optimized, equal, and single-model weights

The optimized blend should be judged primarily by its class-balanced log loss, which is the fitting objective. Macro F1 and accuracy are reported as descriptive diagnostics. An individual model can receive a zero weight when it adds no probability-level information on the external dataset.


In [ ]:
equal_weights = np.full(len(MODEL_ORDER), 1.0 / len(MODEL_ORDER))
optimized_probs = blend_probabilities(optimized_weights)
equal_probs = blend_probabilities(equal_weights)

comparison_rows = []
for name, probs in all_probabilities.items():
    comparison_rows.append(metric_row(name, probs, y_true, sample_weights))
comparison_rows.append(metric_row("equal_weight_ensemble", equal_probs, y_true, sample_weights))
comparison_rows.append(metric_row("optimized_ensemble", optimized_probs, y_true, sample_weights))

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    "log_loss", ascending=True
).reset_index(drop=True)
display(comparison_df.style.format({
    "accuracy": "{:.4f}",
    "macro_f1": "{:.4f}",
    "weighted_f1": "{:.4f}",
    "log_loss": "{:.4f}",
}))

optimized_predictions = optimized_probs.argmax(axis=1)
print("Calibration-data classification report (not final test results):")
print(classification_report(
    y_true,
    optimized_predictions,
    labels=np.arange(len(MODEL_CLASS_NAMES)),
    target_names=MODEL_CLASS_NAMES,
    digits=4,
    zero_division=0,
))


In [ ]:
# Vẽ trọng số tối ưu
plt.figure(figsize=(7, 4.5))
bars = plt.bar(MODEL_ORDER, optimized_weights, color=["#2E86AB", "#F18F01", "#4CAF50"])
plt.ylim(0, max(1.0, float(optimized_weights.max()) * 1.15))
plt.ylabel("Weight")
plt.title("Optimized Ensemble Weights")
for bar, value in zip(bars, optimized_weights):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.015,
        f"{value:.4f}",
        ha="center",
    )
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "optimized_weights.png", dpi=200, bbox_inches="tight")
plt.show()

# Vẽ confusion matrix trên dữ liệu dùng tính weight
cm = confusion_matrix(
    y_true,
    optimized_predictions,
    labels=np.arange(len(MODEL_CLASS_NAMES)),
)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=MODEL_CLASS_NAMES,
    yticklabels=MODEL_CLASS_NAMES,
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Optimized Ensemble — External Weight-Fitting Data")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "external_weight_data_confusion_matrix.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


## Save portable outputs

The JSON file is the main artifact to carry into the final evaluation notebook. The probability CSV permits later auditing or alternative weight experiments without running the three neural networks again.


In [ ]:
# Tạo tệp JSON trọng số để dùng nguyên vẹn trên test gốc
weights_payload = {
    "method": "class_balanced_log_loss_slsqp",
    "model_order": MODEL_ORDER,
    "weights": {
        name: float(weight)
        for name, weight in zip(MODEL_ORDER, optimized_weights)
    },
    "weight_sum": float(optimized_weights.sum()),
    "class_names_in_model_output_order": MODEL_CLASS_NAMES,
    "manifest_keep_status": STRICT_KEEP_STATUS,
    "number_of_external_images": int(len(clean_df)),
    "external_class_counts": {
        MODEL_CLASS_NAMES[index]: int(count)
        for index, count in enumerate(class_counts)
    },
    "objective_balanced_log_loss": float(
        balanced_log_loss_objective(optimized_weights)
    ),
    "important_note": (
        "Freeze these weights before evaluating once on the untouched original test split."
    ),
    "efficientnet_class_order_assumption": MODEL_CLASS_NAMES,
}

weights_path = OUTPUT_DIR / "optimized_ensemble_weights.json"
with open(weights_path, "w", encoding="utf-8") as file:
    json.dump(weights_payload, file, indent=2, ensure_ascii=False)

# Lưu xác suất từng lớp của từng mô hình và ensemble
prediction_df = clean_df[[
    "path",
    "relative_path",
    "resolved_path",
    "dataset",
    "split",
    "label",
    "label_normalized",
    "label_idx",
    "overlap_status",
]].copy()

for model_name in MODEL_ORDER:
    probs = all_probabilities[model_name]
    for class_idx, class_name in enumerate(MODEL_CLASS_NAMES):
        prediction_df[f"{model_name}_prob_{class_name}"] = probs[:, class_idx]
    prediction_df[f"{model_name}_pred_idx"] = probs.argmax(axis=1)

for class_idx, class_name in enumerate(MODEL_CLASS_NAMES):
    prediction_df[f"ensemble_prob_{class_name}"] = optimized_probs[:, class_idx]
prediction_df["ensemble_pred_idx"] = optimized_predictions
prediction_df["ensemble_pred_label"] = [
    MODEL_CLASS_NAMES[index] for index in optimized_predictions
]

predictions_path = OUTPUT_DIR / "external_model_probabilities.csv"
prediction_df.to_csv(predictions_path, index=False)

comparison_path = OUTPUT_DIR / "weight_optimization_summary.csv"
comparison_df.to_csv(comparison_path, index=False)

print("Saved outputs:")
for output_path in sorted(OUTPUT_DIR.iterdir()):
    print(f" - {output_path.name}: {output_path.stat().st_size / (1024**2):.2f} MB")


## Locate the untouched original test split

The final evaluation uses only the `test` directory of the original **Chest X-Ray (Pneumonia, Covid-19, Tuberculosis)** dataset shown in the Kaggle input panel. The finder first checks the known dataset path and then searches attached inputs for a `test` directory whose parent also contains `train` and `val`.

This evaluation is valid only if none of the supplied checkpoints was trained or selected using this original test split. No final-test result should be used to modify the frozen weights.


In [ ]:
VALID_IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"
}


def normalize_folder_label(folder_name):
    # Chuẩn hóa tên thư mục lớp theo đúng thứ tự output của checkpoint
    return LABEL_ALIASES.get(str(folder_name).strip().upper())


def is_valid_original_test_directory(test_directory):
    # Dataset gốc phải có đủ train, val, test và bốn thư mục lớp trong test
    test_directory = Path(test_directory)
    if not test_directory.is_dir():
        return False

    parent = test_directory.parent
    if not (parent / "train").is_dir() or not (parent / "val").is_dir():
        return False

    normalized_classes = {
        normalize_folder_label(child.name)
        for child in test_directory.iterdir()
        if child.is_dir()
    }
    normalized_classes.discard(None)
    return normalized_classes == set(MODEL_CLASS_NAMES)


def locate_original_test_directory():
    # Ưu tiên đúng đường dẫn dataset đã biết từ notebook huấn luyện
    preferred = ORIGINAL_DATASET_ROOT / "test"
    if is_valid_original_test_directory(preferred):
        return preferred

    # Tìm dự phòng nếu Kaggle thay đổi cách mount đường dẫn
    candidates = [
        path for path in dirs_by_name.get("test", [])
        if is_valid_original_test_directory(path)
    ]
    candidates = sorted(set(candidates))

    if not candidates:
        raise FileNotFoundError(
            "Cannot locate the original Chest X-Ray test directory. "
            "Attach the original dataset or correct ORIGINAL_DATASET_ROOT."
        )
    if len(candidates) > 1:
        raise RuntimeError(
            f"Multiple valid original test directories found: {candidates}. "
            "Set ORIGINAL_DATASET_ROOT manually."
        )
    return candidates[0]


ORIGINAL_TEST_DIR = locate_original_test_directory()
print("Original test directory:", ORIGINAL_TEST_DIR)


In [ ]:
def build_original_test_dataframe(test_directory):
    # Tạo manifest chỉ từ thư mục test, tuyệt đối không lấy train hoặc val
    rows = []
    for class_directory in sorted(Path(test_directory).iterdir()):
        if not class_directory.is_dir():
            continue

        normalized_label = normalize_folder_label(class_directory.name)
        if normalized_label is None:
            warnings.warn(f"Ignoring unknown test folder: {class_directory.name}")
            continue

        for image_path in sorted(class_directory.rglob("*")):
            if image_path.is_file() and image_path.suffix.lower() in VALID_IMAGE_EXTENSIONS:
                rows.append({
                    "path": str(image_path),
                    "relative_path": str(image_path.relative_to(test_directory)),
                    "resolved_path": str(image_path),
                    "dataset": "original_chest_xray",
                    "split": "test",
                    "label": class_directory.name,
                    "label_normalized": normalized_label,
                    "label_idx": CLASS_TO_IDX[normalized_label],
                })

    dataframe = pd.DataFrame(rows)
    if dataframe.empty:
        raise ValueError(f"No test images found in: {test_directory}")
    if set(dataframe["label_normalized"]) != set(MODEL_CLASS_NAMES):
        raise ValueError("The original test split does not contain all four classes.")
    return dataframe.reset_index(drop=True)


test_df = build_original_test_dataframe(ORIGINAL_TEST_DIR)
print("Original test images:", len(test_df))
display(
    test_df["label_normalized"]
    .value_counts()
    .reindex(MODEL_CLASS_NAMES)
    .to_frame("count")
)

# Lưu danh sách test thực tế để thí nghiệm có thể được kiểm tra lại
test_df.to_csv(OUTPUT_DIR / "original_test_manifest_used.csv", index=False)


## Final inference with frozen weights

The three models now generate probabilities for the original test images using the same model-specific preprocessing as the external dataset. The optimized weights are read back from the saved JSON file and are not recalculated from test labels.


In [ ]:
def predict_test_probabilities(model_name, model, transform):
    cache_path = OUTPUT_DIR / f"original_test_{model_name}_probabilities.npy"

    # Cache test tách riêng hoàn toàn với cache của dữ liệu tính weight
    if cache_path.exists():
        cached = np.load(cache_path)
        if cached.shape == (len(test_df), len(MODEL_CLASS_NAMES)):
            print(f"Loaded cached test probabilities: {cache_path.name}")
            return cached.astype(np.float64)
        warnings.warn(f"Ignoring incompatible test cache: {cache_path}")

    dataset = ExternalCXRDataset(test_df, transform)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
    )

    model = model.to(DEVICE)
    model.eval()
    probabilities = np.zeros(
        (len(test_df), len(MODEL_CLASS_NAMES)), dtype=np.float32
    )

    with torch.inference_mode():
        for batch_number, (images, _, indices) in enumerate(loader, start=1):
            images = images.to(DEVICE, non_blocking=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                logits = model(images)

            batch_probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
            probabilities[indices.numpy()] = batch_probs

            if batch_number % 25 == 0 or batch_number == len(loader):
                print(
                    f"{model_name}: test batch {batch_number}/{len(loader)} "
                    f"({min(batch_number * BATCH_SIZE, len(test_df))}/{len(test_df)} images)"
                )

    if not np.isfinite(probabilities).all():
        raise ValueError(f"{model_name} produced invalid test probabilities.")
    if not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-4):
        raise ValueError(f"{model_name} test probabilities do not sum to one.")

    np.save(cache_path, probabilities)
    model.to("cpu")
    del loader, dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return probabilities.astype(np.float64)


test_probabilities = {}
for model_name, spec in MODEL_SPECS.items():
    print(f"\nRunning final test inference: {model_name}...")
    test_probabilities[model_name] = predict_test_probabilities(
        model_name=model_name,
        model=spec["model"],
        transform=spec["transform"],
    )

test_y_true = test_df["label_idx"].to_numpy(dtype=np.int64)


In [ ]:
# Đọc lại weight đã lưu thay vì tối ưu lại bằng nhãn test
with open(weights_path, "r", encoding="utf-8") as file:
    frozen_weight_payload = json.load(file)

if frozen_weight_payload["model_order"] != MODEL_ORDER:
    raise ValueError("Saved model order does not match the current model order.")

frozen_weights = np.array([
    frozen_weight_payload["weights"][name] for name in MODEL_ORDER
], dtype=np.float64)

if np.any(frozen_weights < 0) or not np.isclose(frozen_weights.sum(), 1.0, atol=1e-8):
    raise ValueError(f"Invalid frozen weights: {frozen_weights}")

print("Frozen weights used for final test:")
for name, weight in zip(MODEL_ORDER, frozen_weights):
    print(f"{name:15s}: {weight:.8f}")

# Trộn xác suất test mà không dùng nhãn test trong bất kỳ phép tối ưu nào
test_probability_stack = np.stack(
    [test_probabilities[name] for name in MODEL_ORDER], axis=0
)
test_optimized_probs = np.tensordot(
    frozen_weights, test_probability_stack, axes=(0, 0)
)
test_equal_probs = np.mean(test_probability_stack, axis=0)

test_evaluation_rows = []
for name in MODEL_ORDER:
    test_evaluation_rows.append(
        metric_row(name, test_probabilities[name], test_y_true)
    )
test_evaluation_rows.append(
    metric_row("equal_weight_ensemble", test_equal_probs, test_y_true)
)
test_evaluation_rows.append(
    metric_row("optimized_ensemble", test_optimized_probs, test_y_true)
)

test_evaluation_df = pd.DataFrame(test_evaluation_rows).sort_values(
    "macro_f1", ascending=False
).reset_index(drop=True)

print("Final results on the untouched original test split:")
display(test_evaluation_df.style.format({
    "accuracy": "{:.4f}",
    "macro_f1": "{:.4f}",
    "weighted_f1": "{:.4f}",
    "log_loss": "{:.4f}",
}))


## Final classification report and confusion matrix

The following report is the final optimized-ensemble evaluation on the original test split. Unlike the earlier external-data diagnostics, these values may be reported as final results if the test split was never used during training, checkpoint selection, or weight optimization.


In [ ]:
test_optimized_predictions = test_optimized_probs.argmax(axis=1)

final_report_dict = classification_report(
    test_y_true,
    test_optimized_predictions,
    labels=np.arange(len(MODEL_CLASS_NAMES)),
    target_names=MODEL_CLASS_NAMES,
    digits=4,
    zero_division=0,
    output_dict=True,
)
final_report_df = pd.DataFrame(final_report_dict).transpose()

print("Final optimized-ensemble classification report:")
display(final_report_df.style.format("{:.4f}"))

final_cm = confusion_matrix(
    test_y_true,
    test_optimized_predictions,
    labels=np.arange(len(MODEL_CLASS_NAMES)),
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    final_cm,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=MODEL_CLASS_NAMES,
    yticklabels=MODEL_CLASS_NAMES,
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Frozen Optimized Ensemble — Original Test Split")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "original_test_optimized_ensemble_confusion_matrix.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


In [ ]:
# Lưu toàn bộ xác suất và dự đoán test để phục vụ kiểm tra kết quả
final_prediction_df = test_df.copy()

for model_name in MODEL_ORDER:
    probs = test_probabilities[model_name]
    for class_idx, class_name in enumerate(MODEL_CLASS_NAMES):
        final_prediction_df[f"{model_name}_prob_{class_name}"] = probs[:, class_idx]
    final_prediction_df[f"{model_name}_pred_idx"] = probs.argmax(axis=1)

for class_idx, class_name in enumerate(MODEL_CLASS_NAMES):
    final_prediction_df[f"ensemble_prob_{class_name}"] = test_optimized_probs[:, class_idx]

final_prediction_df["ensemble_pred_idx"] = test_optimized_predictions
final_prediction_df["ensemble_pred_label"] = [
    MODEL_CLASS_NAMES[index] for index in test_optimized_predictions
]
final_prediction_df["ensemble_correct"] = (
    final_prediction_df["ensemble_pred_idx"] == final_prediction_df["label_idx"]
)

final_prediction_df.to_csv(
    OUTPUT_DIR / "original_test_model_probabilities.csv", index=False
)
test_evaluation_df.to_csv(
    OUTPUT_DIR / "original_test_evaluation_summary.csv", index=False
)
final_report_df.to_csv(
    OUTPUT_DIR / "original_test_optimized_ensemble_classification_report.csv"
)

# Lưu kết quả cuối ở dạng JSON dễ sử dụng cho báo cáo hoặc notebook khác
final_result_payload = {
    "test_directory": str(ORIGINAL_TEST_DIR),
    "number_of_test_images": int(len(test_df)),
    "class_names": MODEL_CLASS_NAMES,
    "frozen_weights": {
        name: float(weight) for name, weight in zip(MODEL_ORDER, frozen_weights)
    },
    "metrics": {
        row["model"]: {
            key: float(value)
            for key, value in row.items()
            if key != "model"
        }
        for row in test_evaluation_rows
    },
}

with open(
    OUTPUT_DIR / "original_test_final_results.json", "w", encoding="utf-8"
) as file:
    json.dump(final_result_payload, file, indent=2, ensure_ascii=False)

print("All final outputs:")
for output_path in sorted(OUTPUT_DIR.iterdir()):
    print(f" - {output_path.name}: {output_path.stat().st_size / (1024**2):.2f} MB")


## Experiment complete

The notebook has now completed both stages without retuning on the original test labels:

1. Fit and freeze ensemble weights using only strict-clean external images.
2. Evaluate the three individual models, equal-weight ensemble, and frozen optimized ensemble on only the original `test` split.

The main final-result files are `original_test_evaluation_summary.csv`, `original_test_optimized_ensemble_classification_report.csv`, `original_test_model_probabilities.csv`, and `original_test_final_results.json`.
